# Parkinson's Disease Classification - End-to-End Demo

This notebook demonstrates the complete pipeline for PD classification using spiral drawing data.

## Overview
1. Load tabular and image data
2. Extract features (TSFresh + pretrained ResNet)
3. Train traditional ML models
4. Evaluate with confidence intervals
5. Visualize results

**Designed for small datasets (~70 samples)**

## 1. Setup and Imports

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.append('..')

from src.data import load_tabular_data, load_image_data
from src.data.tabular_features import extract_tabular_features
from src.data.image_features import extract_image_features
from src.models import XGBoostClassifier, RandomForestClassifier, EarlyFusion, LateFusion
from src.evaluation import (
    calculate_all_metrics,
    calculate_metrics_with_confidence_intervals,
    print_metrics_report,
    plot_roc_curve,
    plot_confusion_matrix,
    plot_feature_importance,
    cross_validate_model
)

print("✓ All imports successful")

## 2. Configuration

In [ ]:
# Data paths (update these to your data locations)
TABULAR_PATH = "../data/tabular/"
IMAGE_PATH = "../data/images/"

# Random seed for reproducibility
RANDOM_STATE = 42

# Test size
TEST_SIZE = 0.2

# Cross-validation folds
N_SPLITS = 5

print(f"Configuration:")
print(f"  Random seed: {RANDOM_STATE}")
print(f"  Test size: {TEST_SIZE}")
print(f"  CV folds: {N_SPLITS}")

## 3. Load Data

In [ ]:
# Load tabular time-series data
print("Loading tabular data...")
X_tab_train, y_tab_train, X_tab_test, y_tab_test = load_tabular_data(
    TABULAR_PATH,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print("\n" + "="*60)

# Load image data
print("Loading image data...")
X_img_train, y_img_train, X_img_test, y_img_test, img_metadata = load_image_data(
    IMAGE_PATH,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print(f"\nImage classes: {img_metadata['class_names']}")

## 4. Feature Extraction

In [ ]:
# Extract tabular features (TSFresh + manual PD-specific features)
print("Extracting tabular features...")
X_tab_train_feat, X_tab_test_feat, tab_feature_names = extract_tabular_features(
    X_tab_train, 
    X_tab_test, 
    y_tab_train,
    use_tsfresh=True,
    use_manual=True,
    n_features=50  # Limit TSFresh features
)

print(f"Tabular features: {len(tab_feature_names)}")

In [ ]:
# Extract image features using pretrained ResNet
print("Extracting image features...")
try:
    X_img_train_feat, X_img_test_feat, img_feature_dim = extract_image_features(
        X_img_train,
        X_img_test,
        model_name='resnet18',
        batch_size=16
    )
    print(f"Image features: {img_feature_dim}")
except ImportError:
    print("PyTorch not available. Using HOG features instead.")
    from src.data.image_features import TraditionalImageFeatures
    
    extractor = TraditionalImageFeatures()
    X_img_train_feat = extractor.extract_hog_features(X_img_train)
    X_img_test_feat = extractor.extract_hog_features(X_img_test)
    img_feature_dim = X_img_train_feat.shape[1]
    print(f"HOG features: {img_feature_dim}")

## 5. Single Modality Models

### 5.1 Tabular Data Only

In [ ]:
print("Training XGBoost on tabular features...")
tab_model = XGBoostClassifier(
    n_estimators=50,
    max_depth=3,
    random_state=RANDOM_STATE
)
tab_model.fit(X_tab_train_feat, y_tab_train)

# Predictions
y_tab_pred = tab_model.predict(X_tab_test_feat)
y_tab_proba = tab_model.predict_proba(X_tab_test_feat)

# Metrics
tab_metrics = calculate_metrics_with_confidence_intervals(
    y_tab_test, y_tab_pred, y_tab_proba,
    n_bootstrap=1000,
    confidence_level=0.95
)
print_metrics_report(tab_metrics, title="Tabular Model Performance")

In [ ]:
# Feature importance
feature_importance = tab_model.get_feature_importance()
plot_feature_importance(
    feature_importance,
    tab_feature_names,
    model_name="XGBoost (Tabular)",
    top_n=20,
    save_path="../results/figures/tabular_feature_importance.png"
)

### 5.2 Image Data Only

In [ ]:
print("Training XGBoost on image features...")
img_model = XGBoostClassifier(
    n_estimators=50,
    max_depth=3,
    random_state=RANDOM_STATE
)
img_model.fit(X_img_train_feat, y_img_train)

# Predictions
y_img_pred = img_model.predict(X_img_test_feat)
y_img_proba = img_model.predict_proba(X_img_test_feat)

# Metrics
img_metrics = calculate_metrics_with_confidence_intervals(
    y_img_test, y_img_pred, y_img_proba,
    n_bootstrap=1000,
    confidence_level=0.95
)
print_metrics_report(img_metrics, title="Image Model Performance")

## 6. Multi-Modal Fusion

### 6.1 Early Fusion (Combine features)

In [ ]:
print("Training Early Fusion model...")
early_fusion = EarlyFusion(
    model_type='xgboost',
    normalize_features=True
)
early_fusion.fit(X_tab_train_feat, X_img_train_feat, y_tab_train)

# Predictions
y_early_pred = early_fusion.predict(X_tab_test_feat, X_img_test_feat)
y_early_proba = early_fusion.predict_proba(X_tab_test_feat, X_img_test_feat)

# Metrics
early_metrics = calculate_metrics_with_confidence_intervals(
    y_tab_test, y_early_pred, y_early_proba,
    n_bootstrap=1000,
    confidence_level=0.95
)
print_metrics_report(early_metrics, title="Early Fusion Performance")

### 6.2 Late Fusion (Combine predictions)

In [ ]:
print("Training Late Fusion model...")
late_fusion = LateFusion(
    tabular_model_type='xgboost',
    image_model_type='xgboost',
    fusion_method='weighted_average'
)
late_fusion.fit(X_tab_train_feat, X_img_train_feat, y_tab_train)

# Predictions
y_late_pred = late_fusion.predict(X_tab_test_feat, X_img_test_feat)
y_late_proba = late_fusion.predict_proba(X_tab_test_feat, X_img_test_feat)

# Metrics
late_metrics = calculate_metrics_with_confidence_intervals(
    y_tab_test, y_late_pred, y_late_proba,
    n_bootstrap=1000,
    confidence_level=0.95
)
print_metrics_report(late_metrics, title="Late Fusion Performance")

## 7. Model Comparison

In [ ]:
# Compare all models
all_results = {
    'Tabular Only': tab_metrics,
    'Image Only': img_metrics,
    'Early Fusion': early_metrics,
    'Late Fusion': late_metrics
}

from src.evaluation.metrics import compare_models
compare_models(all_results, metric='f1')

## 8. Visualizations

In [ ]:
# ROC curves comparison
plot_multiple_roc_curves(
    {
        'Tabular': (y_tab_test, y_tab_proba),
        'Image': (y_img_test, y_img_proba),
        'Early Fusion': (y_tab_test, y_early_proba),
        'Late Fusion': (y_tab_test, y_late_proba)
    },
    save_path="../results/figures/roc_comparison.png"
)

In [ ]:
# Confusion matrix for best model
plot_confusion_matrix(
    y_tab_test,
    y_early_pred,
    class_names=['Healthy', "Parkinson's"],
    model_name='Early Fusion',
    normalize=True,
    save_path="../results/figures/confusion_matrix.png"
)

## 9. Cross-Validation

For robust performance estimates on small datasets.

In [ ]:
# Combine train and test for CV
X_tab_all = np.vstack([X_tab_train_feat, X_tab_test_feat])
X_img_all = np.vstack([X_img_train_feat, X_img_test_feat])
y_all = np.concatenate([y_tab_train, y_tab_test])

# Perform CV on early fusion model
cv_model = EarlyFusion(model_type='xgboost')
cv_scores = cross_validate_model(
    cv_model,
    X_tab_all,
    y_all,
    X_image=X_img_all,
    n_splits=N_SPLITS,
    random_state=RANDOM_STATE,
    verbose=True
)

## 10. Summary and Conclusions

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT SUMMARY")
print("="*80)
print(f"\nDataset size: {len(y_all)} samples")
print(f"Class distribution: {np.bincount(y_all)}")
print(f"\nTabular features: {len(tab_feature_names)}")
print(f"Image features: {img_feature_dim}")

print("\nBest Model:")
print(f"  Early Fusion F1: {early_metrics['f1']['value']:.4f} "
      f"(95% CI: [{early_metrics['f1']['ci_lower']:.4f}, {early_metrics['f1']['ci_upper']:.4f}])")

print("\nKey Findings:")
print("  1. Traditional ML works well on small datasets")
print("  2. Multi-modal fusion improves performance")
print("  3. Bootstrap CIs show metric uncertainty")

print("\nNext Steps:")
print("  - Try different model architectures")
print("  - Experiment with feature selection")
print("  - Test on external validation set")
print("\n" + "="*80)

---

**Note**: This pipeline is specifically designed for small datasets. Avoid:
- Complex deep learning architectures
- Heavy data augmentation
- Too many features relative to samples

**Recommended for small data**:
- Traditional ML (XGBoost, RF, SVM)
- Proper cross-validation
- Confidence intervals for metrics
- Simple, interpretable models